In [0]:
-- Elimina registros de fact_flights que ya no existen en la tabla silver (mantiene idempotencia)
-- Esto evita acumulación de datos históricos cuando silver se recarga completamente
DELETE FROM airline_catalog.gold.fact_flights
WHERE flight_id NOT IN (
  SELECT flight_id 
  FROM airline_catalog.silver.flights_silver
);

In [0]:
-- Inserta o actualiza los datos en la tabla fact_flights desde la tabla silver y las tablas dim para asegurar idempotencia
MERGE INTO airline_catalog.gold.fact_flights AS f
USING (
  SELECT
    s.flight_id, -- Identificador único del vuelo
    d.date_id, -- Fecha del vuelo
    a.airline_id, -- Aerolínea operadora
    ao.airport_sk AS origin_airport_sk, -- Aeropuerto de origen
    ad.airport_sk AS destination_airport_sk, -- Aeropuerto de destino
    s.flight_number, -- Número comercial del vuelo
    s.tail_number, -- Identificador de la aeronave
    s.scheduled_departure_time, -- Hora programada de salida
    s.actual_departure_time, -- Hora real de salida
    s.departure_delay, -- Retraso en salida
    s.scheduled_arrival_time, -- Hora programada de llegada
    s.actual_arrival_time, -- Hora real de llegada
    s.arrival_delay, -- Retraso en llegada
    s.cancelled, -- Indica si el vuelo fue cancelado
    s.cancellation_code, -- Motivo de cancelación
    s.diverted, -- Indica si el vuelo fue desviado
    s.actual_elapsed_time, -- Duración total del vuelo
    s.air_time, -- Tiempo real en aire
    s.distance, -- Distancia recorrida
    s.carrier_delay, -- Retraso causado por la aerolínea
    s.weather_delay, -- Retraso por clima
    s.nas_delay, -- Retraso por sistema aéreo/nacional
    s.security_delay, -- Retraso por seguridad
    s.late_aircraft_delay, -- Retraso por llegada tardía de aeronave anterior
    s.flight_status, -- Estado del vuelo
    current_timestamp() AS _processing_timestamp -- Timestamp de carga del registro
  FROM airline_catalog.silver.flights_silver s -- Tabla silver de vuelos
  LEFT JOIN airline_catalog.gold.dim_date d ON s.flight_date = d.date_id -- Unión con dimensión fecha
  LEFT JOIN airline_catalog.gold.dim_airline a ON s.airline_code = a.airline_code -- Unión con dimensión aerolínea
  LEFT JOIN airline_catalog.gold.dim_airport ao ON s.origin_code = ao.airport_code -- Unión con dimensión aeropuerto de origen
  LEFT JOIN airline_catalog.gold.dim_airport ad ON s.destination_code = ad.airport_code -- Unión con dimensión aeropuerto de destino
) AS src
ON f.flight_id = src.flight_id
WHEN MATCHED THEN
  UPDATE SET
    f.date_id = src.date_id,
    f.airline_id = src.airline_id,
    f.origin_airport_sk = src.origin_airport_sk,
    f.destination_airport_sk = src.destination_airport_sk,
    f.flight_number = src.flight_number,
    f.tail_number = src.tail_number,
    f.scheduled_departure_time = src.scheduled_departure_time,
    f.actual_departure_time = src.actual_departure_time,
    f.departure_delay = src.departure_delay,
    f.scheduled_arrival_time = src.scheduled_arrival_time,
    f.actual_arrival_time = src.actual_arrival_time,
    f.arrival_delay = src.arrival_delay,
    f.cancelled = src.cancelled,
    f.cancellation_code = src.cancellation_code,
    f.diverted = src.diverted,
    f.actual_elapsed_time = src.actual_elapsed_time,
    f.air_time = src.air_time,
    f.distance = src.distance,
    f.carrier_delay = src.carrier_delay,
    f.weather_delay = src.weather_delay,
    f.nas_delay = src.nas_delay,
    f.security_delay = src.security_delay,
    f.late_aircraft_delay = src.late_aircraft_delay,
    f.flight_status = src.flight_status,
    f._processing_timestamp = src._processing_timestamp
WHEN NOT MATCHED THEN
  INSERT (
    flight_id,
    date_id,
    airline_id,
    origin_airport_sk,
    destination_airport_sk,
    flight_number,
    tail_number,
    scheduled_departure_time,
    actual_departure_time,
    departure_delay,
    scheduled_arrival_time,
    actual_arrival_time,
    arrival_delay,
    cancelled,
    cancellation_code,
    diverted,
    actual_elapsed_time,
    air_time,
    distance,
    carrier_delay,
    weather_delay,
    nas_delay,
    security_delay,
    late_aircraft_delay,
    flight_status,
    _processing_timestamp
  )
  VALUES (
    src.flight_id,
    src.date_id,
    src.airline_id,
    src.origin_airport_sk,
    src.destination_airport_sk,
    src.flight_number,
    src.tail_number,
    src.scheduled_departure_time,
    src.actual_departure_time,
    src.departure_delay,
    src.scheduled_arrival_time,
    src.actual_arrival_time,
    src.arrival_delay,
    src.cancelled,
    src.cancellation_code,
    src.diverted,
    src.actual_elapsed_time,
    src.air_time,
    src.distance,
    src.carrier_delay,
    src.weather_delay,
    src.nas_delay,
    src.security_delay,
    src.late_aircraft_delay,
    src.flight_status,
    src._processing_timestamp
  );

In [0]:
select count(*) from airline_catalog.gold.fact_flights;
